# 01 · Ingestão Bronze — Acidentes ANTT

| | |
|---|---|
| **Fonte** | Portal de Dados Abertos ANTT |
| **Destino** | `Files/bronze/acidentes/` |
| **Frequência** | Sob demanda / agendado via Pipeline |
| **Idempotente** | Sim — pula arquivos já existentes por padrão |

**Fluxo:** Download HTTP → `/tmp/` (driver Spark) → `notebookutils.fs.cp` → OneLake

> **Pré-requisito:** Adicione o `lakehouse_antt` como lakehouse padrão no painel esquerdo antes de executar.

## 1 · Imports

In [ ]:
import os
import time
from dataclasses import dataclass
from typing import List, Set

import requests

## 2 · Parâmetros

> Célula marcada como **Parameters** — valores podem ser sobrescritos por um Data Pipeline.

In [ ]:
NOTEBOOK_NAME:      str  = "ingestao_bronze"
DESTINO_LAKEHOUSE:  str  = "Files/bronze/acidentes"
TMP_DIR:            str  = "/tmp/antt_acidentes"
FORCAR_REDOWNLOAD:  bool = False
TIMEOUT_SEGUNDOS:   int  = 60
LOG_LEVEL:          str  = "INFO"

## 3 · Configuração Compartilhada

In [ ]:
%run 00_nb_config

## 4 · Catálogo de fontes

In [ ]:
@dataclass(frozen=True)
class FonteCSV:
    concessionaria: str
    url: str

    @property
    def filename(self) -> str:
        return self.url.split("/")[-1]


CATALOGO: List[FonteCSV] = [
    FonteCSV("RODOVIA DO AÇO",             "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/2d52007c-b838-4060-a94e-76cd4c51b457/download/demostrativo_acidentes_aco.csv"),
    FonteCSV("AUTOPISTA FLUMINENSE",        "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/4ec93f83-e1a3-4f02-9da4-f135a389a600/download/demostrativo_acidentes_af.csv"),
    FonteCSV("AUTOPISTA FERNÃO DIAS",       "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/359e38d3-e081-4b11-bb2f-0d0670bef9db/download/demostrativo_acidentes_afd.csv"),
    FonteCSV("AUTOPISTA LITORAL SUL",       "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/f8da002f-c9fe-46ad-a59a-7d343d3c965d/download/demostrativo_acidentes_als.csv"),
    FonteCSV("AUTOPISTA PLANALTO SUL",      "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/5dbcd614-3626-4799-ab94-23dcb894468c/download/demostrativo_acidentes_aps.csv"),
    FonteCSV("AUTOPISTA REGIS BITTENCOURT", "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/c23e9f36-fcba-4125-999c-b5bd68304112/download/demostrativo_acidentes_arb.csv"),
    FonteCSV("CONCEBRA",                    "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/7f733042-7a6c-49c7-934f-c2a1cddc5199/download/demostrativo_acidentes_concebra.csv"),
    FonteCSV("CONCER",                      "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/b51768cb-58f8-41b0-99aa-e7221dc5f8cd/download/demostrativo_acidentes_concer.csv"),
    FonteCSV("CRO",                         "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/da18a65d-2c4c-48c1-ba77-8fdd8647ec6d/download/demostrativo_acidentes_cro.csv"),
    FonteCSV("ECO050",                      "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/5a25133c-1871-4341-be2c-91555258b3b4/download/demostrativo_acidentes_eco050.csv"),
    FonteCSV("ECO101",                      "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/6e184bbe-d81d-4de3-830a-19fc0af4e697/download/demostrativo_acidentes_eco101.csv"),
    FonteCSV("ECOPONTE",                    "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/413912fe-55e4-45cc-8692-7a79c087668d/download/demostrativo_acidentes_ecoponte.csv"),
    FonteCSV("ECORIOMINAS",                 "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/d2dea7ff-429f-4c4e-b2d8-68c4f5a673b3/download/demostrativo_acidentes_ecoriominas.csv"),
    FonteCSV("ECOSUL",                      "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/9fc3b057-b822-4bc0-944d-e98a7242bd0e/download/demostrativo_acidentes_ecosul.csv"),
    FonteCSV("ECOVIAS DO ARAGUAIA",         "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/bdada29c-4e4f-4969-8dde-ea619923204b/download/demostrativo_acidentes_ecoviasaraguaia.csv"),
    FonteCSV("ECOVIAS CAPIXABA",            "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/14875e46-9ec5-4b8a-b420-7afa33999a81/download/demostrativo_acidentes_ecoviascapixaba.csv"),
    FonteCSV("ECOVIAS DO CERRADO",          "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/8a4cf3c5-4b36-45a2-bcca-c3f987f6527f/download/demostrativo_acidentes_ecoviasdocerrado.csv"),
    FonteCSV("ELOVIAS",                     "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/656f3908-bc59-47dd-b132-0d2fb8b73ac1/download/demostrativo_acidentes_elovias.csv"),
    FonteCSV("EPR IGUACU",                  "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/c8749e98-1012-4d9e-a9e4-2b2ca581b98d/download/demostrativo_acidentes_epr_iguacu.csv"),
    FonteCSV("EPR LITORAL PIONEIRO",        "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/12994116-56b5-47a6-8de6-fefb7cf8ff03/download/demostrativo_acidentes_epr_litoral_pioneiro.csv"),
    FonteCSV("NOVA 381",                    "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/5d0809f2-e62d-4260-8d2c-dd284d64ea44/download/demostrativo_acidentes_nova_381.csv"),
    FonteCSV("PANTANAL",                    "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/32419d69-184a-499a-a981-2ac5c68d697b/download/demostrativo_acidentes_pantanal.csv"),
    FonteCSV("PRVIAS",                      "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/d280c523-eaef-4d4d-9c08-1b8b6267ddcf/download/demostrativo_acidentes_prvias.csv"),
    FonteCSV("RIOSP",                       "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/0d180b06-fd26-4ecc-a77c-5bbb50c486a6/download/demostrativo_acidentes_riosp.csv"),
    FonteCSV("ROTA VERDE GOIÁS",            "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/f2cd88ea-83ff-4766-b8da-e737a7c36874/download/demostrativo_acidentes_rota-verde-goias.csv"),
    FonteCSV("TRANSBRASILIANA",             "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/17eee041-50cf-4133-9bfa-72314fb435b0/download/demostrativo_acidentes_trans.csv"),
    FonteCSV("VIA040",                      "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/eb737d6c-a96a-4148-964b-9edd516edd14/download/demostrativo_acidentes_via040.csv"),
    FonteCSV("VIA ARAUCARIA",               "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/aa60ce3a-033a-4864-81dc-ae32bea866e5/download/demostrativo_acidentes_viaaraucaria.csv"),
    FonteCSV("VIABAHIA",                    "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/b6ffaaf0-f7a2-45a3-b76d-cafedc86f43b/download/demostrativo_acidentes_viabahia.csv"),
    FonteCSV("VIABRASIL",                   "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/e8644845-a6e5-43fa-8f90-19ab81b2d1f1/download/demostrativo_acidentes_viabrasil.csv"),
    FonteCSV("VIACOSTEIRA",                 "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/2e7f8360-4c5f-4980-827b-93e46f016607/download/demostrativo_acidentes_viacosteira.csv"),
    FonteCSV("VIACRISTAIS",                 "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/db8003a9-f949-4f7b-a5dd-c86c981bc9db/download/demostrativo_acidentes_viacristais.csv"),
    FonteCSV("VIA MINEIRA",                 "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/7c3a9f7a-21fe-4a50-b1a9-93d0b76f3f00/download/demostrativo_acidentes_viamineira.csv"),
    FonteCSV("VIASUL",                      "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/940fa31c-e30e-4a06-9953-4c8b491b2887/download/demostrativo_acidentes_viasul.csv"),
    FonteCSV("WAY 262",                     "https://dados.antt.gov.br/dataset/ef0171a8-f0df-4817-a4ed-b4ff94d87194/resource/98cfbaba-e019-4b77-8401-64e2da096545/download/demostrativo_acidentes_way_262.csv"),
]

log.info("%d fontes no catálogo", len(CATALOGO))

## 5 · Funções

In [ ]:
def validar_lakehouse() -> None:
    """Garante que o lakehouse padrão está acessível antes de iniciar."""
    try:
        notebookutils.fs.ls("Files/")
        log.info("Lakehouse acessível.")
    except Exception as exc:
        raise RuntimeError(
            "Lakehouse não encontrado. Adicione o lakehouse_antt como padrão "
            "no painel esquerdo do notebook antes de executar."
        ) from exc


def preparar_diretorios() -> Set[str]:
    """Cria pastas necessárias e retorna o conjunto de arquivos já existentes no destino."""
    os.makedirs(TMP_DIR, exist_ok=True)
    notebookutils.fs.mkdirs(DESTINO_LAKEHOUSE)
    try:
        existentes = {item.name for item in notebookutils.fs.ls(DESTINO_LAKEHOUSE)}
    except Exception:
        existentes = set()
    log.info("Destino: %s | Arquivos existentes: %d", DESTINO_LAKEHOUSE, len(existentes))
    return existentes


def _download(fonte: FonteCSV) -> bytes:
    """Executa o download HTTP e retorna o conteúdo binário."""
    resp = requests.get(fonte.url, timeout=TIMEOUT_SEGUNDOS)
    resp.raise_for_status()
    return resp.content


def _copiar_para_lakehouse(tmp_path: str, dest_path: str, filename: str, existentes: Set[str]) -> None:
    """Remove arquivo anterior se necessário e copia de /tmp/ para o Lakehouse."""
    if FORCAR_REDOWNLOAD and filename in existentes:
        try:
            notebookutils.fs.rm(dest_path)
        except Exception:
            pass
    notebookutils.fs.cp(f"file://{tmp_path}", dest_path)


def processar_fonte(fonte: FonteCSV, idx: int, total: int, existentes: Set[str]) -> dict:
    """Processa uma única fonte: decide skip/download, executa e retorna o resultado."""
    prefix = f"[{idx:02d}/{total}]"

    if not FORCAR_REDOWNLOAD and fonte.filename in existentes:
        log.info("%s SKIP  %s", prefix, fonte.filename)
        return {"status": "skip", "concessionaria": fonte.concessionaria, "filename": fonte.filename, "kb": 0, "segundos": 0}

    tmp_path  = os.path.join(TMP_DIR, fonte.filename)
    dest_path = f"{DESTINO_LAKEHOUSE}/{fonte.filename}"
    inicio    = time.monotonic()

    try:
        conteudo = _download(fonte)

        with open(tmp_path, "wb") as fh:
            fh.write(conteudo)

        _copiar_para_lakehouse(tmp_path, dest_path, fonte.filename, existentes)

        kb      = round(len(conteudo) / 1024, 1)
        elapsed = round(time.monotonic() - inicio, 1)
        log.info("%s OK    %s (%s KB, %ss)", prefix, fonte.filename, kb, elapsed)
        return {"status": "ok", "concessionaria": fonte.concessionaria, "filename": fonte.filename, "kb": kb, "segundos": elapsed}

    except Exception as exc:
        elapsed = round(time.monotonic() - inicio, 1)
        log.error("%s ERRO  %s — %s", prefix, fonte.filename, exc)
        return {"status": "erro", "concessionaria": fonte.concessionaria, "filename": fonte.filename, "kb": 0, "segundos": elapsed}

    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

## 6 · Execução

In [ ]:
validar_lakehouse()
existentes = preparar_diretorios()

resultados = [
    processar_fonte(fonte, idx, len(CATALOGO), existentes)
    for idx, fonte in enumerate(CATALOGO, 1)
]

## 7 · Relatório

In [ ]:
ok_list   = [r for r in resultados if r["status"] == "ok"]
skip_list = [r for r in resultados if r["status"] == "skip"]
erro_list = [r for r in resultados if r["status"] == "erro"]

total_mb  = round(sum(r["kb"] for r in ok_list) / 1024, 1)
total_seg = round(sum(r["segundos"] for r in ok_list), 1)

log.info("Baixados: %d | Pulados: %d | Erros: %d | Volume: %s MB | Tempo: %ss",
         len(ok_list), len(skip_list), len(erro_list), total_mb, total_seg)

if erro_list:
    log.error("Arquivos com erro:")
    for r in erro_list:
        log.error("  - %s (%s)", r["filename"], r["concessionaria"])

arquivos_destino = sorted(notebookutils.fs.ls(DESTINO_LAKEHOUSE), key=lambda x: x.name)
log.info("%d arquivo(s) em %s:", len(arquivos_destino), DESTINO_LAKEHOUSE)
for item in arquivos_destino:
    log.info("  %s (%s KB)", item.name, round(item.size / 1024, 1))

# Retorna status para o Pipeline — falha se houver erros
if erro_list:
    notebookutils.notebook.exit(f"ERRO: {len(erro_list)} arquivo(s) falharam.")
else:
    notebookutils.notebook.exit(f"OK: {len(ok_list)} baixados, {len(skip_list)} pulados.")